# Notebook 1: Einführung und Daten

**Statistik für Data Science** · Kurs 3,906 · Bachelor Computer Science · HS2026
Universität St. Gallen

---
### Projektteam:
- Andrin Greitmann
- Noah Appenzeller
- Manuel Schmid

---

Dieses Notebook führt die Inhalte von **VL01 (Einführung & Daten)** in Python aus. 

## 0. Setup
Importiert `pandas`, `numpy`, `seaborn` und `matplotlib` und setzt den Seed.

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Seed: Startwert für den Zufallsgenerator. Gleicher Seed bei jedem Durchlauf damit Resultate deterministisch & reproduzierbar sind. 
# Den Wert selbst haben wir beliebig gewählt, wir haben ihn als erstes vor der Analyse gewählt und danach nicht mehr geändert.
# Der Seed wird für alles Zufällige in allen Notebooks verwendet.
SEED = 2001
RNG = np.random.default_rng(SEED)

# Plot-Defauls
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7.5, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

In [4]:
import sys

print("Python  ", sys.version.split()[0])
for modul in (np, pd, sns):
    print(f"{modul.__name__:8s}", modul.__version__)
print("Seed    ", SEED)

Python   3.14.4
numpy    2.5.3
pandas   3.0.6
seaborn  0.13.2
Seed     2001


### Unseren Datensatz laden

In [5]:
from ucimlrepo import fetch_ucirepo

# Datensatz direkt aus dem UCI Repository laden (ID 117 = Census-Income KDD)
census_income_kdd = fetch_ucirepo(id=117)

# Etwas spezielles an unserem Datenset: ucimlrepo liefert die Daten aufgeteilt in Features und Target. 
# (Wahrscheinlich für Machine Learning Projekte/Algorithmen)
# Target ist das Einkommen (über/unter 50K), Features sind alle anderen Merkmale.
X = census_income_kdd.data.features
y = census_income_kdd.data.targets

# Für die EDA wollten wir alle Spalten in einem DataFrame damit das Einkommen mit den anderen Merkmalen verglichen werden kann.
# join() funktioniert hier zum Glück recht einfach, weil X und y dieselben Zeilen in derselben Reihenfolge haben.
census = X.join(y)

# Metadaten: Herkunft, Beschreibung und Erhebung des Datensatzes (Damit wir auf Bias analysieren können)
print("Metadata: " + str(census_income_kdd.metadata))

print(f"Rows: {census.shape[0]}, Columns: {census.shape[1]}")

# Variablenbeschreibung: Name, Typ und Bedeutung jeder Spalte (Grundlage um später Messniveau zu bestimmen)
print("Variable descriptions:")

display(census_income_kdd.variables)

Metadata: {'uci_id': 117, 'name': 'Census-Income (KDD)', 'repository_url': 'https://archive.ics.uci.edu/dataset/117/census+income+kdd', 'data_url': 'https://archive.ics.uci.edu/static/public/117/data.csv', 'abstract': 'This data set contains weighted census data extracted from the 1994 and 1995 current population surveys conducted by the U.S. Census Bureau.', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 299285, 'num_features': 41, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Education Level', 'Marital Status', 'Race', 'Sex', 'Other', 'Nationality', 'Income'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2000, 'last_updated': 'Fri Mar 08 2024', 'dataset_doi': '10.24432/C5N30T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set contains weighted census data extracted

,name,role,type,demographic,description,units,missing_values
0,AAGE,Feature,Integer,Age,age,None,no
1,ACLSWKR,Feature,Categorical,NaN,class of worker,None,no
2,ADTINK,Feature,Integer,NaN,industry code,None,no
3,ADTOCC,Feature,Integer,Occupation,occupation code,None,no
4,AHGA,Feature,Integer,Education Level,education,None,no
5,AHSCOL,Feature,Categorical,Education Level,enrolled in edu last week,None,no
6,AMARITL,Feature,Categorical,Marital Status,marital status,None,no
7,AMJIND,Feature,Categorical,NaN,major industry code,None,no
8,AMJOCC,Feature,Categorical,Occupation,major occupation code,None,no
9,ARACE,Feature,Categorical,Race,race,None,no


## 1. Datentypen und Messniveaus
Erster Blick auf unseren Datensatz:

In [6]:
census.info()

<class 'pandas.DataFrame'>
RangeIndex: 199523 entries, 0 to 199522
Data columns (total 42 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   AAGE       199523 non-null  int64  
 1   ACLSWKR    199523 non-null  str    
 2   ADTINK     199523 non-null  int64  
 3   ADTOCC     199523 non-null  int64  
 4   AHGA       199523 non-null  str    
 5   AHSCOL     199523 non-null  str    
 6   AMARITL    199523 non-null  str    
 7   AMJIND     199523 non-null  str    
 8   AMJOCC     199523 non-null  str    
 9   ARACE      199523 non-null  str    
 10  AREORGN    199523 non-null  str    
 11  ASEX       199523 non-null  str    
 12  AUNMEM     199523 non-null  str    
 13  AUNTYPE    199523 non-null  str    
 14  AWKSTAT    199523 non-null  str    
 15  CAPGAIN    199523 non-null  int64  
 16  GAPLOSS    199523 non-null  int64  
 17  DIVVAL     199523 non-null  int64  
 18  FILESTAT   199523 non-null  str    
 19  GRINREG    199523 non-null  str   

### Wichtigste Erkenntnisse zur Datenqualität und Vollständigkeit

1. **Nicht jede Spalte ist vollständig:**
Ein signifikanter Teil des Datensatzes weist Lücken auf. Wir haben hier drei Spalten als Beispiele herausgesucht:
    - **`MIGMTR1`** (Migration Code - Change in MSA) hat nur 99'827 von 199'523 Einträgen (es fehlen knapp 50 % der Daten).
    - **`PEFNTVTY`** (Country of Birth Father) hat 192'810 von 199'523 Einträgen (es fehlen 6.713 Werte).
    - **`GRINST`** (State of Previous Residence) hat 198'815 von 199'523 Einträgen (es fehlen 708 Werte).

2. **Systematische Lücken bei migrationsspezifischen Daten:**
Auffällig ist, dass die Spalten `MIGMTR1`, `MIGMTR3`, `MIGMTR4` und `MIGSUN` alle exakt dieselbe Anzahl an nicht-leeren Werten (99'827) aufweisen. Dies deutet stark darauf hin, dass diese Werte nicht zufällig fehlen sondern aufgrund einer gemeinsamen Abhängigkeit. 

3. **Fehlerhafte Metadaten-Dokumentation (`missing_values` Spalte):**
Ein Problem erkannten wir bei der Dokumentationsspalte `missing_values`. Dort wird bei einigen Spalten fälschlicherweise der Wert `no` angegeben, obwohl `df.info()` eindeutig Lücken belegt. Zum Beispiel die Spalte `PENATVTY`, bei der 3'393 Werte fehlen. Man darf sich bei der Vorverarbeitung also nicht blind auf die textuelle Dokumentation verlassen.

4. **Der `dtype` ist nicht das Messniveau:** Zum Beispiel ist `ADTOCC` als `int64` typisiert, aber es handelt sich um eine nominale Skala (Kategoriencodes für Berufe ohne jegliche Rangfolge). Hier würde Pandas zwar einen Mittelwert berechnen, dies wäre mathematisch aber nicht zulässig, da es keinen „durchschnittlichen Beruf“ gibt.

### Spaltenübersicht

Die folgende Tabelle erstellt einen Überblick über alle 42 Spalten in unserem Datensatz inklusive einiger Beispiele:

In [7]:
def spalten_ueberblick(df: pd.DataFrame, beispiele: int = 3) -> pd.DataFrame:
    """Kompakter Steckbrief je Spalte: dtype, Anzahl eindeutiger Werte, fehlende Werte, Beispiele.

    Erster Schritt bei jedem unbekannten Datensatz.
    """
    zeilen = []
    for spalte in df.columns:
        s = df[spalte]
        werte = s.dropna().unique()[:beispiele]
        zeilen.append(
            {
                "dtype": str(s.dtype),
                "eindeutig": s.nunique(dropna=True),
                "fehlend": int(s.isna().sum()),
                "fehlend_%": round(s.isna().mean() * 100, 1),
                "beispiele": ", ".join(map(str, werte)),
            }
        )
    return pd.DataFrame(zeilen, index=df.columns)


spalten_ueberblick(census)

,dtype,eindeutig,fehlend,fehlend_%,beispiele
AAGE,int64,91,0,0.0,"73, 58, 18"
ACLSWKR,str,9,0,0.0,"Not in universe, Self-employed-not incorpora..."
ADTINK,int64,52,0,0.0,"0, 4, 40"
ADTOCC,int64,47,0,0.0,"0, 34, 10"
AHGA,str,17,0,0.0,"High school graduate, Some college but no de..."
AHSCOL,str,3,0,0.0,"Not in universe, High school, College or un..."
AMARITL,str,7,0,0.0,"Widowed, Divorced, Never married"
AMJIND,str,24,0,0.0,"Not in universe or children, Construction, ..."
AMJOCC,str,15,0,0.0,"Not in universe, Precision production craft ..."
ARACE,str,5,0,0.0,"White, Asian or Pacific Islander, Amer Indi..."


### Die vier Messniveaus

| Skala | Eigenschaft | Erlaubte Statistik |
|---|---|---|
| **Nominal** | Kategorien ohne Ordnung | Modus, Häufigkeiten |
| **Ordinal** | Rangfolge, Abstände undefiniert | zusätzlich Median, Perzentile |
| **Intervall** | gleiche Abstände, kein natürlicher Nullpunkt | zusätzlich Mittelwert, Standardabweichung |
| **Ratio** | absoluter Nullpunkt | zusätzlich Verhältnisse, geometrisches Mittel |

In [8]:
# Inhaltliche Zuordnung. 
MESSNIVEAU = {
    "AAGE": "ratio",          # 0 Jahre ist ein echter Nullpunkt
    "ACLSWKR": "nominal",     # Arbeiterklasse (z.B. Private, Federal Gov)
    "ADTINK": "nominal",      # Industrie-Code, int64 aber reiner Code ohne Rang
    "ADTOCC": "nominal",      # Berufs-Code, int64 aber reiner Code ohne Rang
    "AHGA": "ordinal",        # Bildungsabschluss (hat eine logische Rangfolge)
    "AHSCOL": "nominal",      # Einschreibung Schule/Uni
    "AMARITL": "nominal",     # Familienstand
    "AMJIND": "nominal",      # Hauptbranche
    "AMJOCC": "nominal",      # Hauptberufsgruppe
    "ARACE": "nominal",       # Ethnie/Rasse
    "AREORGN": "nominal",     # Hispanische Herkunft
    "ASEX": "nominal",        # Geschlecht
    "AUNMEM": "nominal",      # Gewerkschaftsmitgliedschaft
    "AUNTYPE": "nominal",     # Grund für Arbeitslosigkeit
    "AWKSTAT": "nominal",     # Vollzeit-/Teilzeit-Status
    "CAPGAIN": "ratio",       # Kapitalgewinne (0 $ ist ein echter Nullpunkt)
    "GAPLOSS": "ratio",       # Kapitalverluste (0 $ ist ein echter Nullpunkt)
    "DIVVAL": "ratio",        # Dividenden aus Aktien (0 $ ist ein echter Nullpunkt)
    "FILESTAT": "nominal",    # Steuerstatus
    "GRINREG": "nominal",     # Region des vorherigen Wohnsitzes
    "GRINST": "nominal",      # Bundesstaat des vorherigen Wohnsitzes
    "HHDFMX": "nominal",      # Detaillierte Haushaltsstruktur
    "HHDREL": "nominal",      # Beziehung im Haushalt
    "MARSUPWRT": "ratio",     # Gewichtung / Instance Weight
    "MIGMTR1": "nominal",     # Migrationscode (MSA)
    "MIGMTR3": "nominal",     # Migrationscode (Region)
    "MIGMTR4": "nominal",     # Migrationscode (innerhalb Region)
    "MIGSAME": "nominal",     # Gleiches Haus wie vor 1 Jahr (Ja/Nein-Code)
    "MIGSUN": "nominal",      # Migration (Sunbelt)
    "NOEMP": "ratio",         # Anzahl der Beschäftigten beim Arbeitgeber
    "PARENT": "nominal",      # Elternpräsenz im Haushalt
    "PEFNTVTY": "nominal",    # Geburtsland Vater
    "PEMNTVTY": "nominal",    # Geburtsland Mutter
    "PENATVTY": "nominal",    # Geburtsland Selbst
    "PRCITSHP": "nominal",    # Staatsbürgerschaft
    "SEOTR": "nominal",       # Selbstständigkeit (Code)
    "VETQVA": "nominal",      # Fragebogen für Veteranen
    "VETYN": "nominal",       # Veteranenstatus (Ja/Nein)
    "WKSWORK": "ratio",       # Gearbeitete Wochen im Jahr (0 ist echter Nullpunkt)
    "AHRSPAY": "ratio",       # Stundenlohn (0 $ ist echter Nullpunkt)
    "year": "intervall",      # Jahreszahlen haben keinen natürlichen Nullpunkt
    "income": "ordinal",      # Einkommensklasse (z.B. <50k, >=50k), hat Rangfolge
}

zuordnung = pd.DataFrame(
    {
        "dtype": census.dtypes.astype(str),
        "messniveau": pd.Series(MESSNIVEAU),
    }
)
zuordnung["mittelwert_zulaessig"] = zuordnung["messniveau"].isin(["intervall", "ratio"])
zuordnung["mittelwert"] = [
    census[spalte].mean() if zulaessig else "-"
    for spalte, zulaessig in zuordnung["mittelwert_zulaessig"].items()
]
zuordnung

,dtype,messniveau,mittelwert_zulaessig,mittelwert
AAGE,int64,ratio,True,34.494199
ACLSWKR,str,nominal,False,-
ADTINK,int64,nominal,False,-
ADTOCC,int64,nominal,False,-
AHGA,str,ordinal,False,-
AHSCOL,str,nominal,False,-
AMARITL,str,nominal,False,-
AMJIND,str,nominal,False,-
AMJOCC,str,nominal,False,-
ARACE,str,nominal,False,-


Wie unsere Übersicht zeigt, führt der Datentyp `int64` dazu, dass `pandas` mathematisch völlig unzulässige Mittelwerte für rein nominale Variablen (wie z.B. den Berufscode `ADTOCC`) berechnen würde. Es gibt jedoch auch Spalten wie z.B. `CAPGAIN` (Kapitalgewinne), die den `int64` Datentyp haben, bei denen ein Mittelwert durchaus sinnvoll ist. 

---

## 2. Ursprung der Daten und Bias

### Wie sind unsere Daten entstanden?

Um die vier Fragen zu dem Datenset (aus dem Referenznotebook) zu beantworten haben wir die Dokumentation auf UCI und die Beschreibung der zugrundeliegenden Umfrage des U.S. Census Bureau angeschaut.

**1. Wer ist in den Daten und wer fehlt?**
Die Daten stammen aus dem "Current Population Survey (CPS)" der Jahre 1994 und 1995. Befragt werden Haushalte, die Zielpopulation ist die zivile Bevölkerung der USA, die nicht in Einrichtungen lebt. Also fehlen grundsätzlich alle Personen, die in Gefängnissen, Pflegeheimen oder Kasernen leben. Auch zum Beispiel Obdachlose ohne festen Wohnsitz fehlen. Kinder sind hingegen enthalten, obwohl viele Fragen für sie gar nicht gelten/sinnvoll sind (daher die vielen "Not in universe"-Werte).

**2. Gibt es einen Selektionsprozess?**
Ja, gibt es:
- Die Teilnahme am CPS ist freiwillig. Haushalte, die nicht antworten, fehlen. Ebenfalls werden fehlende Einkommensangaben vom Census Bureau teilweise imputiert.
- Die Stichprobe ist geschichtet, darum gibt es das Gewicht `MARSUPWRT` (siehe nächster Abschnitt).

**3. Wie wurde gemessen?**
Alle Angaben sind Selbstauskünfte aus Interviews. Oft beantwortet eine Person die Fragen für den ganzen Haushalt. Beim Einkommen erwarten wir deshalb eher zu tiefe Werte, weil Einkommen anderer Haushaltsmitglieder oder Kapitalerträge oft nicht genau bekannt sind. Zusätzlich werden sehr hohe Einkommen im öffentlichen CPS aus Datenschutzgründen nach oben begrenzt, was hohe Werte ebenfalls nach unten verzerrt.

**4. Gibt es unmögliche oder verdächtige Werte?**
<br>Das haben wir direkt im Code unten geprüft:

In [9]:
# Kinder im Datensatz (CPS-Einkommensfragen gelten erst ab 15 Jahren)
print(f"Anteil unter 15 Jahren: {(census['AAGE'] < 15).mean():.1%}")
print(f"Alter min/max: {census['AAGE'].min()} / {census['AAGE'].max()}\n")

# Doppelte Zeilen (ohne Gewicht, da dieses pro Person verschieden sein kann)
print(f"Doppelte Zeilen: {census.drop(columns='MARSUPWRT').duplicated().sum()}\n")

# Fehlen die Migrationswerte in einem bestimmten Jahr?
print("Anteil fehlend MIGMTR1 pro Jahr:")
print(census.groupby("year")["MIGMTR1"].apply(lambda s: s.isna().mean()), "\n")

# Platzhalter "Not in universe" in den Textspalten
text_spalten = census.select_dtypes(exclude="number")
not_in_universe = text_spalten.apply(lambda s: s.str.strip().str.startswith("Not in universe").mean() * 100)
print("Anteil 'Not in universe' in %:")
print(not_in_universe[not_in_universe > 0].sort_values(ascending=False).round(1), "\n")

# Nullen und Maxima bei den Geldvariablen
geld = ["AHRSPAY", "CAPGAIN", "GAPLOSS", "DIVVAL"]
print(pd.DataFrame({
    "anteil_null_%": ((census[geld] == 0).mean() * 100).round(1),
    "max": census[geld].max(),
}))

Anteil unter 15 Jahren: 23.8%
Alter min/max: 0 / 90

Doppelte Zeilen: 46627

Anteil fehlend MIGMTR1 pro Jahr:
year
94    0.0
95    1.0
Name: MIGMTR1, dtype: float64 

Anteil 'Not in universe' in %:
VETQVA     99.0
AUNTYPE    97.0
AHSCOL     93.7
GRINREG    92.1
GRINST     92.1
AUNMEM     90.4
PARENT     72.3
MIGSAME    50.7
AMJIND     50.5
AMJOCC     50.5
ACLSWKR    50.2
MIGSUN     42.1
MIGMTR1     0.8
MIGMTR3     0.8
MIGMTR4     0.8
dtype: float64 

         anteil_null_%    max
AHRSPAY           94.3   9999
CAPGAIN           96.3  99999
GAPLOSS           98.0   4608
DIVVAL            89.4  99999


**Erkenntnisse aus der Werte-Prüfung:**

- **Kinder machen einen grossen Teil aus.** 23.8 % der Personen sind jünger als 15 Jahre, das jüngste Alter ist 0. Für diese Personen sind Fragen zu Arbeit und Einkommen gar nicht gestellt worden. Das Höchstalter liegt bei genau 90, was darauf hindeutet, dass auch das Alter nach oben eventuell begrenzt wurde.

- **Viele Duplikate.** 46'627 Zeilen sind (ohne Gewicht) identisch mit einer anderen Zeile. Aber da wir sehr viele grobe Kategorien haben muss das nicht zwingend ein Fehler sein, da zwei verschiedene Personen schon durchaus dieselben Merkmale haben können.

- **Die Migrationsdaten fehlen für 1995 komplett.** `MIGMTR1` fehlt 1994 nie und 1995 immer. Also fehlen die Werte nicht zufällig, sondern wurden 1995 schlicht nicht erhoben oder nicht mitgeliefert. Aussagen zur Migration gelten für uns also nur für 1994.

- **"Not in universe" ist oft der Normalfall.** Bei `VETQVA`, `AUNTYPE` und `AHSCOL` betrifft der Platzhalter über 90 % der vorhandenen Werte, bei Beruf und Branche (`AMJOCC`, `AMJIND`, `ACLSWKR`) rund die Hälfte. Diese Werte sind keine echte Kategorie, sondern bedeuten, dass die Frage nicht gestellt wurde. Dies reduziert die vorhandene Datenmenge über die entsprechenden Kategorien stark. 

- **Verdächtige/fehldende Geldvariablen.** Zwischen 89 % und 98 % der Werte sind 0, wahrscheinlich meistens weil die Person gar kein solches Einkommen hat. Uns sind auch die Maxima aufgefallen: 99'999 bei `CAPGAIN` und `DIVVAL` sowie 9'999 bei `AHRSPAY` sind sehr wahrscheinlich Obergrenzen und somit vermutlich keine echten Beträge.

**Fazit:** Ein grosser Teil unseres Datensatzes besteht aus Werten, die zwar nicht als fehlend markiert sind, inhaltlich aber "nicht zutreffend" bedeuten. Ungefilterte Mittelwerte würden dadurch stark nach unten gezogen. Für jede weitere Kennzahl legen wir deshalb fest, auf welche Teilpopulation sie sich bezieht, z.B. nur Erwerbstätige oder nur Personen ab 15 Jahren.

### Erkenntnis: Der Datensatz ist gewichtet

Die Daten stammen nicht aus einer einfachen Zufallsstichprobe. Der Census Bureau hat geschichtet gezogen (stratified sampling) und manche Gruppen absichtlich häufiger befragt, als es ihrem Anteil in der Bevölkerung entspricht, damit auch über kleine Gruppen genug Daten vorhanden sind. Diese Gruppen sind in unserem Datensatz deshalb übervertreten.

Ausgeglichen wird das durch die Spalte `MARSUPWRT`. Sie gibt an, wie viele Personen in der Bevölkerung eine Zeile repräsentiert. Eine Zeile mit Gewicht 2000 steht für 2000 Personen, eine mit Gewicht 500 nur für 500. In einem gewichteten Mittelwert zählt die erste Person also viermal so stark.

Also bedeutet das für unsere Analyse:
- Ungewichtete Kennzahlen beschreiben nur die Stichprobe. Für die EDA konnten wir sie zwar verwenden, haben sie aber entsprechend gekennzeichnet.
- Für Aussagen über die Population gewichten wir die Werte mit `MARSUPWRT`.

Nachfolgend haben wir ungewichtete und mit den Census-Gewichten berechnete Mittelwerte verglichen und untersucht, wie sich die Gewichtung auf Kennzahlen und Einkommensanteile auswirkt:

In [10]:
spalten = ["AAGE", "WKSWORK", "AHRSPAY", "CAPGAIN", "DIVVAL"]
gewicht = census["MARSUPWRT"]

vergleich = pd.DataFrame(
    {
        "ungewichtet": census[spalten].mean(),
        "gewichtet": [np.average(census[s], weights=gewicht) for s in spalten],
    }
)
vergleich["abweichung_%"] = (vergleich["gewichtet"] / vergleich["ungewichtet"] - 1) * 100

# Auch Anteile verschieben sich: z.B. Anteil Personen mit Einkommen über 50k
ueber_50k = census["income"].str.strip() == "50000+."
print(f"Anteil >50k ungewichtet: {ueber_50k.mean():.2%}")
print(f"Anteil >50k gewichtet:   {np.average(ueber_50k, weights=gewicht):.2%}\n")

vergleich.round(2)

Anteil >50k ungewichtet: 6.21%
Anteil >50k gewichtet:   6.41%



,ungewichtet,gewichtet,abweichung_%
AAGE,34.49,34.47,-0.06
WKSWORK,23.17,23.58,1.76
AHRSPAY,55.43,57.37,3.50
CAPGAIN,434.72,441.56,1.57
DIVVAL,197.53,197.52,-0.01


Der Vergleich zeigt, dass die Unterschiede zwischen gewichteten und ungewichteten Werten bei unseren Variablen klein sind. Beim Alter (`AAGE`) und bei den Dividenden (`DIVVAL`) ist die Abweichung praktisch 0%. Am grössten ist sie beim Stundenlohn (`AHRSPAY`) mit 3.5%. Auch der Anteil der Personen mit einem Einkommen über 50k steigt nur leicht von 6.21% auf 6.41%.

Die gewichteten Werte liegen dort, wo es eine Abweichung gibt, jeweils etwas höher. Das deutet darauf hin, dass Gruppen mit tieferem Lohn und weniger Arbeitswochen in der Stichprobe leicht übervertreten sind. Für diese Kennzahlen auf Ebene des gesamten Datensatzes verfälscht die Schichtung das Bild also fast nicht. Trotzdem verwenden wir für Aussagen über die Population die gewichteten Werte, da sich die Abweichungen bei Auswertungen einzelner Untergruppen verstärken könnten.

### Sampling Bias

Um darzustellen, dass eine Stichprobe die Zielpopulation nicht (zwingend) akkurat abbilet haben wir hier dargestellt, was passiert, wenn man nur eine bestimmte Gruppe der Bevölkerung befragt. Hier ein Beispiel für das Alter von Personen mit einem bestimmten Familienstand.

In [11]:
wahrer_mittelwert = np.average(census["AAGE"], weights=census["MARSUPWRT"])

nach_familienstand = census.groupby("AMARITL")["AAGE"].agg(["size", "mean"])
nach_familienstand["abweichung"] = nach_familienstand["mean"] - wahrer_mittelwert
nach_familienstand = nach_familienstand.sort_values("abweichung")

print(f"Mittleres Alter, alle Personen: {wahrer_mittelwert:.2f}\n")
print(nach_familienstand)

Mittleres Alter, alle Personen: 34.47

                                  size       mean  abweichung
AMARITL                                                      
Never married                    86485  15.922622  -18.551058
Married-A F spouse present         665  31.673684   -2.799996
Separated                         3460  41.760116    7.286435
Married-spouse absent             1518  45.525692   11.052011
Divorced                         12710  46.129583   11.655903
Married-civilian spouse present  84222  46.705433   12.231753
Widowed                          10463  71.750454   37.276774


Hier ist ersichtlich, dass je nachdem, welche Gruppe man befragt, das Durchschnittsalter stark vom wahren Wert abweicht. Bei "Never married" ist der Mittelwert deutlich tiefer, weil in dieser Gruppe auch alle Kinder enthalten sind. Bei "Widowed" liegt er klar darüber, weil Verwitwete meist älter sind. Wenn man also z.B. eine Umfrage nur über einen Kanal macht, der vor allem eine bestimmte Gruppe erreicht, zieht man danach falsche Schlüsse über die ganze Population.

---

## 4. Erste EDA

>EDA(Exploratory Data Analysis) heisst Muster erkennen, Fehler aufspüren und Hypothesen bilden. EDA erzeugt Hypothesen, sie bestätigt sie nicht.

Wir gehen in der Reihenfolge der Vorlesung vor: zuerst die Struktur, dann die Kennzahlen, dann die Bilder.

### 4.1 Struktur

Die Spalten haben wir in Abschnitt 1 schon einzeln angeschaut (`info()`, Spaltenübersicht, Messniveaus). Jetzt kommt der Blick auf die **Zeilen**: Wie sieht eine einzelne Person in unserem Datensatz aus?

In [12]:
print(f"Form des Datensatzes: {census.shape[0]} Zeilen, {census.shape[1]} Spalten\n")

# Bei 42 Spalten kürzt pandas die Anzeige mit "..." in der Mitte.
# option_context hebt diese Grenze nur für diese eine Ausgabe auf, die globalen Einstellungen bleiben gleich.
# .T dreht die Tabelle (Spalten werden zu Zeilen), so passen alle 42 Merkmale untereinander auf den Bildschirm.
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(census.head().T)

Form des Datensatzes: 199523 Zeilen, 42 Spalten



,0,1,2,3,4
AAGE,73,58,18,9,10
ACLSWKR,Not in universe,Self-employed-not incorporated,Not in universe,Not in universe,Not in universe
ADTINK,0,4,0,0,0
ADTOCC,0,34,0,0,0
AHGA,High school graduate,Some college but no degree,10th grade,Children,Children
AHSCOL,Not in universe,Not in universe,High school,Not in universe,Not in universe
AMARITL,Widowed,Divorced,Never married,Never married,Never married
AMJIND,Not in universe or children,Construction,Not in universe or children,Not in universe or children,Not in universe or children
AMJOCC,Not in universe,Precision production craft & repair,Not in universe,Not in universe,Not in universe
ARACE,White,White,Asian or Pacific Islander,White,White


`head()` zeigt immer die ersten fünf Zeilen der Datei. Das ist nur dann ein fairer Einblick, wenn die Datei nicht sortiert ist (z.B. nach Jahr oder Alter). Deshalb prüfen wir das kurz und ziehen zusätzlich eine Zufallsstichprobe. Mit `random_state=SEED` ist sie bei jedem Durchlauf dieselbe.

In [13]:
# Ist die Datei sortiert? Wenn ja, wären die ersten Zeilen nicht typisch für den Rest.
print("Jahr der ersten 10 Zeilen: ", census["year"].head(10).tolist())
print("Alter der ersten 10 Zeilen:", census["AAGE"].head(10).tolist(), "\n")

# Zufallsstichprobe statt der ersten Zeilen, reproduzierbar dank Seed.
stichprobe = census.sample(5, random_state=SEED)

with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(stichprobe.T)

Jahr der ersten 10 Zeilen:  [95, 94, 95, 94, 94, 95, 94, 95, 95, 94]
Alter der ersten 10 Zeilen: [73, 58, 18, 9, 10, 48, 42, 28, 47, 34] 



,64609,186849,67714,75989,32858
AAGE,13,23,42,24,13
ACLSWKR,Not in universe,Private,Private,Private,Not in universe
ADTINK,0,33,14,11,0
ADTOCC,0,19,13,22,0
AHGA,Children,Some college but no degree,Masters degree(MA MS MEng MEd MSW MBA),Some college but no degree,Children
AHSCOL,Not in universe,Not in universe,Not in universe,Not in universe,Not in universe
AMARITL,Never married,Never married,Divorced,Never married,Never married
AMJIND,Not in universe or children,Retail trade,Manufacturing-durable goods,Manufacturing-durable goods,Not in universe or children
AMJOCC,Not in universe,Sales,Technicians and related support,Adm support including clerical,Not in universe
ARACE,White,White,White,White,White


Schon in diesen zehn Zeilen fallen ein paar Dinge auf (Details in den Erkenntnissen unten). Was in fünf Zeilen auffällt, ist aber erst ein **Verdacht**. Bevor wir es als Erkenntnis aufschreiben, zählen wir über alle Zeilen nach:

In [14]:
# 1) "Not in universe under 1 year old" bei MIGSAME: wirklich nur Säuglinge?
print(pd.crosstab(census["MIGSAME"], census["year"]), "\n")

# 2) Passt die Kategorie "Children or Armed Forces" bei AWKSTAT zu den gearbeiteten Wochen?
wochen = (census["WKSWORK"] > 0).map({False: "0 Wochen", True: "mind. 1 Woche"})
print(pd.crosstab(census["AWKSTAT"], wochen), "\n")

# 3) Heisst AHRSPAY = 0 wirklich "kein Lohn"? Anteil Nullen bei allen, die 52 Wochen gearbeitet haben.
ganzjahr = census[census["WKSWORK"] == 52]
print(f"Personen mit 52 Arbeitswochen: {len(ganzjahr)}, davon mit AHRSPAY = 0: {(ganzjahr['AHRSPAY'] == 0).mean():.1%}\n")

# 4) Ist NOEMP eine Anzahl Personen? Verteilung bei Beschäftigten in der Industrie.
industrie = census["AMJIND"].str.strip().str.startswith("Manufacturing")
print("NOEMP in der Industrie:")
print(census.loc[industrie, "NOEMP"].value_counts().sort_index())

year                                 94     95
MIGSAME                                       
No                                15773      0
Not in universe under 1 year old   1516  99696
Yes                               82538      0 

WKSWORK                             0 Wochen  mind. 1 Woche
AWKSTAT                                                    
Children or Armed Forces               71727          52042
Full-time schedules                      926          39810
Not in labor force                     22446           4362
PT for econ reasons usually FT             8            517
PT for econ reasons usually PT            23           1186
PT for non-econ reasons usually FT        87           3235
Unemployed full-time                     520           1791
Unemployed part- time                    246            597 

Personen mit 52 Arbeitswochen: 70314, davon mit AHRSPAY = 0: 89.0%

NOEMP in der Industrie:
NOEMP
0     436
1    1283
2    1091
3    2349
4    3335
5    1172
6  

Zum Schluss der Teil, den die Vorlesung unter "Struktur: Datentypen" führt: Wie verteilen sich unsere 42 Spalten auf Speichertyp (`dtype`) und Messniveau? Dafür nehmen wir die Zuordnung aus Abschnitt 1 als Kreuztabelle.

In [15]:
# Zeilen = Speichertyp, Spalten = Messniveau aus Abschnitt 1, "total" = Randsummen.
pd.crosstab(zuordnung["dtype"], zuordnung["messniveau"], margins=True, margins_name="total")

messniveau,intervall,nominal,ordinal,ratio,total
dtype,,,,,
float64,0,0,0,1,1
int64,1,4,0,7,12
str,0,27,2,0,29
total,1,31,2,8,42


**Erkenntnisse aus der Struktur:**

- **Eine Zeile ist eine Person.** 199'523 Personen mit je 42 Merkmalen. Kinder, Rentner und Erwerbstätige stehen gemischt im selben Datensatz.

- **Die Datei ist nicht sortiert.** Jahr (94/95) und Alter wechseln sich schon in den ersten zehn Zeilen ab. `head()` gibt hier also einen fairen ersten Eindruck, die Zufallsstichprobe zeigt dasselbe Bild.

- **"Not in universe" und der Code 0 sind der Normalfall für "Nicht Gefragt".** Bei der 73-jährigen Witwe (Zeile 0) und den beiden Kindern (Zeilen 3 und 4) steht bei fast allen Arbeitsfragen "Not in universe", bei Branche und Beruf (`ADTINK`, `ADTOCC`) steht 0. Die 0 ist dort keine Zahl, sondern ein Platzhalter für "nicht gefragt".

- **Ein Platzhalter mit falscher Bezeichnung.** Bei `MIGSAME` steht auch bei der 73-Jährigen "Not in universe under 1 year old". Die Kreuztabelle zeigt warum: Diesen Wert haben alle 99'696 Personen aus 1995, aber nur 1'516 aus 1994 (das sind echte Säuglinge). Für 1995 heisst der Wert also in Wahrheit "nicht erhoben", passend zu den NaN bei `MIGMTR1`, `MIGMTR3`, `MIGMTR4` und `MIGSUN` aus Abschnitt 2. Diese Werte sind faktisch fehlend, obwohl sie nicht als NaN markiert sind.

- **Die Ausprägung "Children or Armed Forces" bei `AWKSTAT` lässt sich nicht wörtlich lesen.** Die Person mit Index 1 (58 Jahre, selbständig) hat im Jahr 52 Wochen gearbeitet, ist bei `AWKSTAT` aber als "Children or Armed Forces" erfasst, obwohl sie weder ein Kind noch beim Militär ist. Das ist kein Einzelfall: 52'042 Personen mit dieser Ausprägung haben mindestens eine Woche gearbeitet. Warum sie so eingeordnet sind, geht aus der Datensatz-Dokumentation (`census-income.names`) nicht hervor. Für die Analyse heisst das: Wir interpretieren diese Ausprägung nicht als "Kinder und Militär", sondern als ungeklärt. Für die Frage, ob jemand erwerbstätig war, stützen wir uns auf `WKSWORK`.

- **0 bei `AHRSPAY` heisst nicht "kein Lohn".** Die Person in Zeile 1 und drei Erwerbstätige in der Stichprobe haben 51 bis 52 Wochen gearbeitet und trotzdem einen Stundenlohn von 0. Bei allen Personen mit 52 Arbeitswochen sind es 89 %. Die 0 bedeutet also meistens "nicht im Stundenlohn bezahlt" oder "nicht erfasst".

- **`NOEMP` ist keine Anzahl Personen.** Die Spalte hat nur die Werte 0 bis 6, und in der Industrie ist 6 der häufigste Wert. Wäre das eine Anzahl, hätten die grössten Industriebetriebe sechs Mitarbeitende. Es sind also Grössenklassen des Arbeitgebers, `NOEMP` ist damit **ordinal** und nicht ratio. Die Zuordnung in Abschnitt 1 sollten wir entsprechend korrigieren.

- **Der dtype ist nicht das Messniveau, auch in der Summe.** Von den 12 `int64`-Spalten sind 4 nominal (`ADTINK`, `ADTOCC`, `SEOTR`, `VETYN`), dazu kommt `NOEMP`. Für all diese würde `describe()` in 4.2 einen Mittelwert ausgeben, den wir nicht interpretieren dürfen. Ausserdem ist `year` nur als 94/95 gespeichert und `income` als Text "- 50000." bzw. "50000+." (deshalb das `.str.strip()` in Abschnitt 2).

**Hypothesen für die nächsten Schritte** (EDA erzeugt Hypothesen, sie bestätigt sie nicht):
1. Weil 0 und "Not in universe" so häufig sind, ziehen sie die Mittelwerte der Geld- und Arbeitsvariablen über alle Personen stark nach unten. In 4.2 rechnen wir die Kennzahlen deshalb zusätzlich für sinnvolle Teilgruppen (z.B. nur Personen mit Wert > 0).
2. Histogramme ohne Filter (4.3) werden bei diesen Variablen vor allem einen riesigen Balken bei 0 zeigen.
3. Bei den kategorialen Variablen (4.4) wird "Not in universe" oft die häufigste Kategorie (der Modus) sein.

### 4.2 Kennzahlen

Welche Kennzahl zulässig ist, entscheidet das Messniveau (Abschnitt 1). Wir gehen deshalb in drei Schritten vor:

| Messniveau | Kennzahlen | Werkzeug |
|---|---|---|
| metrisch (intervall, ratio) | Mittelwert, Median, Quantile, Standardabweichung | `describe()` |
| ordinal | Median, Quantile (über die Rangfolge) | geordnetes `Categorical` |
| nominal | Modus, Häufigkeiten | `describe(exclude="number")` |

Alle Kennzahlen in diesem Abschnitt sind **ungewichtet**, beschreiben also die Stichprobe. Abschnitt 2 hat gezeigt, dass die Gewichtung auf Ebene des ganzen Datensatzes wenig ändert.

#### Metrische Variablen

`describe()` ohne Auswahl würde alle 13 Zahlenspalten zusammenfassen, auch die Codes wie `ADTINK` oder `VETYN`, deren Mittelwert unzulässig ist. Wir wählen deshalb nur die echten metrischen Merkmale aus.

In [16]:
# Nur echte metrische Merkmale (Messniveau ratio). Bewusst nicht dabei:
# MARSUPWRT (Gewicht, kein Merkmal der Person), year (nur 94/95), NOEMP (Grössenklassen, siehe 4.1)
metrisch = ["AAGE", "WKSWORK", "AHRSPAY", "CAPGAIN", "GAPLOSS", "DIVVAL"]

kennzahlen = census[metrisch].describe().T
# Zwei Zusatzspalten, die describe() nicht liefert:
kennzahlen["anteil_null_%"] = (census[metrisch] == 0).mean() * 100            # wie oft steht genau 0?
kennzahlen["n_am_max"] = (census[metrisch] == census[metrisch].max()).sum()   # wie viele liegen genau auf dem Maximum?
kennzahlen.round(2)

,count,mean,std,min,25%,50%,75%,max,anteil_null_%,n_am_max
AAGE,199523.0,34.49,22.31,0.0,15.0,33.0,50.0,90.0,1.42,725
WKSWORK,199523.0,23.17,24.41,0.0,0.0,8.0,52.0,52.0,48.11,70314
AHRSPAY,199523.0,55.43,274.90,0.0,0.0,0.0,0.0,9999.0,94.33,1
CAPGAIN,199523.0,434.72,4697.53,0.0,0.0,0.0,0.0,99999.0,96.30,390
GAPLOSS,199523.0,37.31,271.90,0.0,0.0,0.0,0.0,4608.0,98.04,4
DIVVAL,199523.0,197.53,1984.16,0.0,0.0,0.0,0.0,99999.0,89.40,25


Wir überprüfen die folgenden Eigenschaften:

- **Signal 1:** `count` kleiner als die Zeilenzahl heisst, es fehlen Werte.
- **Signal 2:** Liegt der Mittelwert weit weg vom Median, ist die Verteilung schief. Faustregel für das Verhältnis Mittelwert/Median: ungefähr 1 = symmetrisch, deutlich über 1 = rechtsschief, unter 1 = linksschief.

Wir rechnen beide nach:

In [17]:
n = len(census)
signale = pd.DataFrame({
    "fehlend (NaN)": n - census[metrisch].count(),
    "mittelwert": census[metrisch].mean(),
    "median": census[metrisch].median(),
})
# Bei Median 0 ist das Verhältnis nicht definiert (Division durch 0), dort steht NaN.
signale["verhaeltnis"] = signale["mittelwert"] / signale["median"].replace(0, np.nan)
signale.round(2)

,fehlend (NaN),mittelwert,median,verhaeltnis
AAGE,0,34.49,33.0,1.05
WKSWORK,0,23.17,8.0,2.90
AHRSPAY,0,55.43,0.0,NaN
CAPGAIN,0,434.72,0.0,NaN
GAPLOSS,0,37.31,0.0,NaN
DIVVAL,0,197.53,0.0,NaN


**Signal 1** schlägt nirgends an, keine metrische Spalte hat ein NaN. In 4.1 haben wir aber gesehen, dass die Lücken bei uns in den Nullen stecken.

**Signal 2** versagt bei vier von sechs Variablen, weil der Median 0 ist. Ein Median von 0 sagt nur: Mehr als die Hälfte der Personen hat hier 0. Deshalb rechnen wir dieselben Kennzahlen noch einmal **nur für Personen mit einem Wert > 0**, also nur für die, die das Merkmal überhaupt haben. Bei `AAGE` ist das nicht nötig, dort ist 0 ein echter Wert (Säuglinge).

In [18]:
def kennzahlen_ohne_null(df: pd.DataFrame, spalten: list[str]) -> pd.DataFrame:
    """Mittelwert und Median nur über Werte > 0, also nur über Personen, die das Merkmal haben."""
    zeilen = {}
    for spalte in spalten:
        positiv = df.loc[df[spalte] > 0, spalte]
        zeilen[spalte] = {
            "n": len(positiv),
            "anteil_%": len(positiv) / len(df) * 100,
            "mittelwert": positiv.mean(),
            "median": positiv.median(),
            "verhaeltnis": positiv.mean() / positiv.median(),
        }
    return pd.DataFrame(zeilen).T.astype({"n": "int64"})


kennzahlen_ohne_null(census, ["WKSWORK", "AHRSPAY", "CAPGAIN", "GAPLOSS", "DIVVAL"]).round(2)

,n,anteil_%,mittelwert,median,verhaeltnis
WKSWORK,103540,51.89,44.66,52.0,0.86
AHRSPAY,11304,5.67,978.32,800.0,1.22
CAPGAIN,7379,3.70,11754.50,5178.0,2.27
GAPLOSS,3906,1.96,1906.03,1887.0,1.01
DIVVAL,21141,10.60,1864.23,301.0,6.19


Bei `AHRSPAY` (Stundenlohn) fällt etwas auf: Der Median der Lohnbezieher liegt bei 800. Als Dollar gelesen wäre das ein mittlerer Stundenlohn von 800 USD, was nicht sein kann. Unsere Vermutung: Der Wert ist in **Cents** gespeichert. Das prüfen wir mit den Quantilen:

In [19]:
# Plausibilitätscheck der Einheit: Ist AHRSPAY in Cents gespeichert?
lohn = census.loc[census["AHRSPAY"] > 0, "AHRSPAY"]
print(lohn.quantile([0.05, 0.25, 0.50, 0.75, 0.95]))

print(f"\nMedian als Dollar gelesen: {lohn.median():.0f} $ pro Stunde")
print(f"Median als Cents gelesen:  {lohn.median() / 100:.2f} $ pro Stunde")

0.05     425.00
0.25     584.75
0.50     800.00
0.75    1200.00
0.95    2000.00
Name: AHRSPAY, dtype: float64

Median als Dollar gelesen: 800 $ pro Stunde
Median als Cents gelesen:  8.00 $ pro Stunde


Fazit: Lohn ist sehr wahrscheinlich in Cents angegeben.

#### Ordinale Variable: Bildung (`AHGA`)

Bei ordinalen Daten sind Median und Quantile zulässig, der Mittelwert nicht. Die Rangfolge müssen wir pandas explizit mitgeben, sonst sortiert es alphabetisch ("10th grade" käme dann vor "1st 2nd 3rd or 4th grade").

Wir betrachten nur Personen **ab 25 Jahren**: Bei Jüngeren läuft die Ausbildung oft noch, und Kinder haben die Kategorie "Children".

Die Reihenfolge ist eine inhaltliche Entscheidung. Zwei Stellen sind nicht eindeutig, und wir haben sie so gesetzt: der berufsbezogene vor dem akademischen Associates-Abschluss und "Prof school degree" (z.B. Medizin, Jura) vor dem Doktorat.

In [20]:
# Rangfolge aufsteigend festlegen.
BILDUNG = [
    "Children",
    "Less than 1st grade",
    "1st 2nd 3rd or 4th grade",
    "5th or 6th grade",
    "7th and 8th grade",
    "9th grade",
    "10th grade",
    "11th grade",
    "12th grade no diploma",
    "High school graduate",
    "Some college but no degree",
    "Associates degree-occup /vocational",
    "Associates degree-academic program",
    "Bachelors degree(BA AB BS)",
    "Masters degree(MA MS MEng MEd MSW MBA)",
    "Prof school degree (MD DDS DVM LLB JD)",
    "Doctorate degree(PhD EdD)",
]

ab25 = census[census["AAGE"] >= 25]
bildung = ab25["AHGA"].str.strip().astype(pd.CategoricalDtype(categories=BILDUNG, ordered=True))

# Median und Quartile laufen über die Rangposition (Codes 0 bis 16), nicht über einen Zahlenwert.
# interpolation="lower" sorgt dafür, dass immer eine existierende Kategorie herauskommt.
raenge = bildung.cat.codes
print(f"Personen ab 25 Jahren: {len(ab25)}\n")
for beschriftung, q in [("25 %", 0.25), ("Median", 0.50), ("75 %", 0.75)]:
    print(f"{beschriftung:7s}", bildung.cat.categories[int(raenge.quantile(q, interpolation="lower"))])
print(f"{'Modus':7s}", bildung.mode()[0])

print("\nVerteilung in Rangfolge (in %):")
print((bildung.value_counts(normalize=True).sort_index() * 100).round(1))

Personen ab 25 Jahren: 125707

25 %    High school graduate
Median  High school graduate
75 %    Associates degree-academic program
Modus   High school graduate

Verteilung in Rangfolge (in %):
AHGA
Children                                   0.0
Less than 1st grade                        0.6
1st 2nd 3rd or 4th grade                   1.4
5th or 6th grade                           2.4
7th and 8th grade                          4.8
9th grade                                  2.5
10th grade                                 3.5
11th grade                                 3.1
12th grade no diploma                      1.0
High school graduate                      34.1
Some college but no degree                17.2
Associates degree-occup /vocational        3.9
Associates degree-academic program         3.2
Bachelors degree(BA AB BS)                14.7
Masters degree(MA MS MEng MEd MSW MBA)     5.2
Prof school degree (MD DDS DVM LLB JD)     1.4
Doctorate degree(PhD EdD)                  1.0
Na

#### Nominale Variablen

`describe(exclude="number")` liefert für Textspalten andere Kennzahlen, weil das Messniveau ein anderes ist: `count`, `unique` (Anzahl Kategorien), `top` (der Modus) und `freq` (wie oft der Modus vorkommt). Dazu rechnen wir den Anteil des Modus und prüfen, ob der Modus eine echte Antwort oder ein Platzhalter ist.

In [21]:
kategorial = census.describe(exclude="number").T
kategorial["top_anteil_%"] = (kategorial["freq"] / kategorial["count"] * 100).astype(float).round(1)

# Ist der Modus ein Platzhalter statt einer echten Antwort? (siehe 4.1)
platzhalter = kategorial["top"].str.contains("Not in universe|Children or Armed Forces")
kategorial["modus_platzhalter"] = platzhalter
print(f"Bei {platzhalter.sum()} von {len(kategorial)} Textspalten ist der Modus ein Platzhalter.\n")

with pd.option_context("display.max_rows", None):
    display(kategorial.sort_values("top_anteil_%", ascending=False))

Bei 13 von 29 Textspalten ist der Modus ein Platzhalter.



,count,unique,top,freq,top_anteil_%,modus_platzhalter
VETQVA,199523,3,Not in universe,197539,99.0,True
AUNTYPE,199523,6,Not in universe,193453,97.0,True
income,199523,2,-50000,187141,93.8,False
AHSCOL,199523,3,Not in universe,186943,93.7,True
GRINST,198815,50,Not in universe,183750,92.4,True
GRINREG,199523,6,Not in universe,183750,92.1,True
AUNMEM,199523,3,Not in universe,180459,90.4,True
PENATVTY,196130,42,United-States,176989,90.2,False
PRCITSHP,199523,5,Native- Born in the United States,176992,88.7,False
AREORGN,199523,10,All other,171907,86.2,False


**Erkenntnisse aus den Kennzahlen** (ungewichtet, also Aussagen über die Stichprobe):

- **Signal 1 (`count` kleiner als die Zeilenzahl, also fehlen Werte) schlägt nicht an, obwohl viele Werte fehlen.** Keine metrische Spalte hat ein NaN. Die Lücken stecken in den Nullen: 48 % bei `WKSWORK`, 89 % bis 98 % bei den Geldvariablen. Hätten wir nur auf `count` geschaut, hätten wir das übersehen.

- **Ein Median von 0 ist kein typischer Wert, sondern ein Mehrheitsbefund.** Bei `AHRSPAY`, `CAPGAIN`, `GAPLOSS` und `DIVVAL` sind alle drei Quartile 0. Ein Mittelwert über alle Personen, z.B. 434.72 USD Kapitalgewinn, beschreibt deshalb niemanden: Die meisten haben 0, und wer etwas hat, hat deutlich mehr.

- **Signal 2 (Vergleich von Mittelwert und Median) kann bei gemischten Gruppen in die Irre führen.** Über alle Personen ergibt `WKSWORK` ein Verhältnis von 2.90, also scheinbar rechtsschief. In Wahrheit besteht die Verteilung aus zwei Gruppen: 48 % mit 0 Wochen (vor allem Kinder und Nichterwerbstätige) und 35 % mit genau 52 Wochen. Ein einzelnes Verhältnis kann so eine Verteilung mit zwei Häufungen nicht beschreiben.

- **Erst die Teilpopulation macht die Form sichtbar.** Nur Personen mit Wert > 0:
  - `DIVVAL`: Mittelwert 1'864 USD, Median 301 USD, Verhältnis 6.19. Stark rechtsschief, wenige sehr hohe Dividenden ziehen den Mittelwert nach oben.
  - `CAPGAIN`: Verhältnis 2.27, ebenfalls deutlich rechtsschief.
  - `GAPLOSS`: Mittelwert und Median fast gleich (1'906 gegen 1'887), ungefähr symmetrisch.
  - `WKSWORK`: Mittelwert 44.66 liegt unter dem Median von 52, also **linksschief**. Wer arbeitet, arbeitet meistens das ganze Jahr, wenige kurze Einsätze ziehen den Mittelwert nach unten.
  - `AAGE` (alle Personen): Mittelwert 34.49 und Median 33 liegen nah beieinander, ungefähr symmetrisch.

- **Die Häufung beim Maximum zeigt, welche Art Grenze vorliegt.**
  - `WKSWORK` = 52 (70'314 Personen): natürliche Obergrenze, ein Jahr hat 52 Wochen.
  - `AAGE` = 90 (725 Personen) und `CAPGAIN` = 99'999 (390 Personen): Häufung am Rand, also **Top-Coding**. Höhere Werte wurden auf diese Grenze gesetzt. Das bestätigt die Vermutung aus Abschnitt 2.
  - `DIVVAL` = 99'999 (25 Personen): vermutlich ebenfalls Top-Coding.
  - `AHRSPAY` = 9'999 (1 Person): keine Häufung, eher ein einzelner Extremwert.

- **`AHRSPAY` ist in Cents gespeichert.** Das 5-%-Quantil der Lohnbezieher liegt bei exakt 425. Das entspricht 4.25 USD, dem US-Mindestlohn von 1994/95. Der Median entspricht 8.00 USD pro Stunde. Der Mittelwert von 55.43 aus Abschnitt 2 heisst also: 55 Cents, gemittelt über alle Personen inklusive Kinder. Ausserdem haben nur 5.7 % überhaupt einen Wert, die Variable taugt also nur für eine kleine Teilgruppe.

- **Bildung (ab 25 Jahren):** Median und Modus sind "High school graduate", das obere Quartil liegt bei "Associates degree-academic program". Dieses Quartil liegt aber knapp an einer Kategoriengrenze (74.5 % kumuliert bis zum berufsbezogenen Associates-Abschluss). Würden wir die beiden Associates-Abschlüsse in umgekehrter Reihenfolge setzen, wäre das Ergebnis ein anderes. Eine ordinale Kennzahl ist nur so belastbar wie die festgelegte Rangfolge.

- **Bei 13 von 29 Textspalten ist der Modus ein Platzhalter** ("Not in universe" oder "Children or Armed Forces"). Hypothese 3 aus 4.1 trifft damit zu. Für diese Spalten ist der Modus keine inhaltliche Aussage. Ein sinnvoller Modus ergibt sich erst innerhalb der Personen, denen die Frage gestellt wurde.

**Hypothesen für 4.3 (Bilder):**
1. Histogramme der Geldvariablen über alle Personen zeigen fast nur einen Balken bei 0. Aussagekräftige Bilder gibt es nur für Werte > 0, bei `DIVVAL` und `CAPGAIN` wegen der starken Rechtsschiefe vermutlich erst auf logarithmischer Skala.
2. `WKSWORK` zeigt zwei Häufungen, bei 0 und bei 52.
3. `AAGE` zeigt eine breite Verteilung mit einer kleinen Spitze bei 90 (Top-Coding).